<a href="https://colab.research.google.com/github/ArthBachhuka123/MachineLearning/blob/main/Pytorch_Optuna.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.9/400.9 kB 13.1 MB/s eta 0:00:00


In [2]:
!pip install kaggle
!mkdir ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

!kaggle datasets download -d zalando-research/fashionmnist

Dataset URL: https://www.kaggle.com/datasets/zalando-research/fashionmnist
License(s): other
  0% 0.00/68.8M [00:00<?, ?B/s]
100% 68.8M/68.8M [00:00<00:00, 1.81GB/s]


In [3]:
import zipfile
import os
zip_path = "/content/fashionmnist.zip"
extract_to = "/content"

with zipfile.ZipFile(zip_path,"r") as zip_ref:
  zip_ref.extractall(extract_to)

In [4]:
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split

torch.manual_seed(41)
data = pd.read_csv("/content/fashion-mnist_train.csv")
test_data = pd.read_csv("/content/fashion-mnist_test.csv")

In [5]:
X = data.drop("label",axis=1)
y = data["label"]
X_train,X_val,y_train,y_val = train_test_split(X,y,test_size=0.2,random_state=42)
X_train = X_train/255
X_val = X_val/255

In [6]:
from torch.utils.data import DataLoader , Dataset
class CustomDataset(Dataset):
  def __init__(self,features,labels):
    self.features = torch.tensor(features.values,dtype=torch.float32)
    self.labels = torch.tensor(labels.values,dtype=torch.long)
  def __len__(self):
    return len(self.features)
  def __getitem__(self,idx):
    return self.features[idx],self.labels[idx]
train_dataset = CustomDataset(X_train,y_train)
val_dataset = CustomDataset(X_val,y_val)
test_dataset = CustomDataset(test_data.drop("label",axis=1),test_data["label"])



In [7]:
import torch.nn as nn
class MyNN(nn.Module):
  def __init__(self,input_dim, output_dim, num_hidden_layers, neuron_per_layer, dropout_rate):
    super().__init__()
    layers = []
    for i in range(num_hidden_layers):
      layers.append(nn.Linear(input_dim,neuron_per_layer))
      layers.append(nn.BatchNorm1d(neuron_per_layer))
      layers.append(nn.ReLU())
      layers.append(nn.Dropout(dropout_rate))
      input_dim = neuron_per_layer

    layers.append(nn.Linear(input_dim,output_dim))
    self.network = nn.Sequential(*layers)

  def forward(self,x):
    return self.network(x)

In [12]:
def objective(trial):
  num_hidden_layers = trial.suggest_int("num_hidden_layers",5,10)
  neuron_per_layer = trial.suggest_int("neuron_per_layer",10,50)
  epochs = trial.suggest_int("epochs",10,25)
  lr = trial.suggest_float("lr",1e-5,1e-1,log=True)
  dropout_rate = trial.suggest_float("dropout_rate",0.2,0.5)
  batch_size = trial.suggest_categorical("batch_size",[32,64,128])
  optimizer = trial.suggest_categorical("optimizer",["Adam","SGD","RMSProp"])
  weight_decay = trial.suggest_float("weight_decay",1e-5,1e-3,log=True)

  model = MyNN(input_dim=784,output_dim=10,num_hidden_layers=num_hidden_layers,neuron_per_layer=neuron_per_layer,dropout_rate=dropout_rate)
  model = model.to("cuda")

  train_loader = DataLoader(train_dataset,batch_size=32,shuffle=True,pin_memory=True)
  val_loader = DataLoader(val_dataset,batch_size=32,shuffle=False,pin_memory=True)
  test_loader = DataLoader(test_dataset,batch_size=32,shuffle=False,pin_memory=True)

  criterion = nn.CrossEntropyLoss()

  if optimizer == "Adam":
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
  elif optimizer == "RMSProp":
    optimizer = torch.optim.RMSprop(model.parameters(), lr=lr, weight_decay=weight_decay)
  elif optimizer == "SGD":
    optimizer = torch.optim.SGD(model.parameters(), lr=lr, weight_decay=weight_decay)

  model.train()
  for epoch in range(1, epochs + 1):
    total_epoch_loss = 0
    for batch_features, batch_labels in train_loader:
        batch_features = batch_features.to("cuda")
        batch_labels = batch_labels.to("cuda")
        outputs = model(batch_features)
        loss = criterion(outputs, batch_labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_epoch_loss += loss.item()


  model.eval()
  accuracy_list = []
  with torch.no_grad():
    for batch_features,batch_labels in val_loader:
      batch_features = batch_features.to("cuda")
      batch_labels = batch_labels.to("cuda")
      output = model(batch_features)
      prediction = torch.argmax(output,dim=1)
      accuracy_list.append((prediction==batch_labels).float().mean())

  return sum(accuracy_list)/len(accuracy_list)


In [13]:
import optuna as optuna
study = optuna.create_study(direction="maximize",sampler=optuna.samplers.TPESampler())
study.optimize(objective, n_trials=5)

[I 2025-11-05 06:00:14,964] A new study created in memory with name: no-name-0ca1af09-801c-4079-b42d-443a69c5723e
[I 2025-11-05 06:02:00,570] Trial 0 finished with value: 0.23624999821186066 and parameters: {'num_hidden_layers': 10, 'neuron_per_layer': 45, 'epochs': 11, 'lr': 0.0003495784244673031, 'dropout_rate': 0.44860397672642216, 'batch_size': 32, 'optimizer': 'Adam', 'weight_decay': 0.00011972829170143097}. Best is trial 0 with value: 0.23624999821186066.
[I 2025-11-05 06:04:01,435] Trial 1 finished with value: 0.38741666078567505 and parameters: {'num_hidden_layers': 6, 'neuron_per_layer': 20, 'epochs': 19, 'lr': 0.000332177919170517, 'dropout_rate': 0.29707454139017886, 'batch_size': 128, 'optimizer': 'SGD', 'weight_decay': 1.1741296461086125e-05}. Best is trial 1 with value: 0.38741666078567505.
[I 2025-11-05 06:05:29,247] Trial 2 finished with value: 0.22091665863990784 and parameters: {'num_hidden_layers': 8, 'neuron_per_layer': 14, 'epochs': 11, 'lr': 0.0016876404501362326,

In [14]:
print(study.best_value)
print(study.best_params)

0.8103333115577698
{'num_hidden_layers': 6, 'neuron_per_layer': 27, 'epochs': 18, 'lr': 4.701568036729881e-05, 'dropout_rate': 0.20061446707698186, 'batch_size': 64, 'optimizer': 'RMSProp', 'weight_decay': 5.447991104292992e-05}


In [18]:
params = study.best_trial.params
best_model = MyNN(
    input_dim=784,
    output_dim=10,
    num_hidden_layers=params["num_hidden_layers"],
    neuron_per_layer=params["neuron_per_layer"],
    dropout_rate=params["dropout_rate"]
)

In [19]:
best_model

MyNN(
  (network): Sequential(
    (0): Linear(in_features=784, out_features=27, bias=True)
    (1): BatchNorm1d(27, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.20061446707698186, inplace=False)
    (4): Linear(in_features=27, out_features=27, bias=True)
    (5): BatchNorm1d(27, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.20061446707698186, inplace=False)
    (8): Linear(in_features=27, out_features=27, bias=True)
    (9): BatchNorm1d(27, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU()
    (11): Dropout(p=0.20061446707698186, inplace=False)
    (12): Linear(in_features=27, out_features=27, bias=True)
    (13): BatchNorm1d(27, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (14): ReLU()
    (15): Dropout(p=0.20061446707698186, inplace=False)
    (16): Linear(in_features=27, out_features=27, bias=True)
    (17): BatchNorm1d